In [2]:
pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Training Model

In [5]:
import cv2
import os
import datetime
import time

def take_img(enrollment, name):
    # Check if the input fields are not empty
    if enrollment == '' or name == '':
        print("Please fill in all fields!")
        return
    
    try:
        # Create a directory for training images if it doesn't exist
        base_dir = 'TrainingImage'
        if not os.path.exists(base_dir):
            os.makedirs(base_dir)

        # Create a subdirectory for the specific enrollment ID
        enrollment_dir = os.path.join(base_dir, enrollment)  # Subfolder named after enrollment ID
        if not os.path.exists(enrollment_dir):
            os.makedirs(enrollment_dir)

        # Open the webcam
        cam = cv2.VideoCapture(0)
        detector = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')  # Load face detection model
        
        sampleNum = 0  # Initialize sample number
        
        print("Starting image capture. Press 'q' to quit.")
        
        while True:
            ret, img = cam.read()  # Read frame from webcam
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
            faces = detector.detectMultiScale(gray, 1.3, 5)  # Detect faces
            
            for (x, y, w, h) in faces:
                cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)  # Draw rectangle around face
                sampleNum += 1
                
                # Save captured face images in the specific enrollment subfolder
                cv2.imwrite(f"{enrollment_dir}/{name}.{enrollment}.{sampleNum}.jpg", gray)
                print(f"Image {sampleNum} saved for Enrollment: {enrollment}, Name: {name}")
                
            cv2.imshow('Frame', img)  # Show the frame with detected faces
            
            if cv2.waitKey(1) & 0xFF == ord('q'):  # Break on 'q' key press
                break
            
            if sampleNum >= 70:  # Stop after capturing a certain number of samples
                break

        cam.release()  # Release the webcam
        cv2.destroyAllWindows()  # Close all OpenCV windows

        ts = time.time()  # Get current timestamp
        Date = datetime.datetime.fromtimestamp(ts).strftime('%Y-%m-%d')  # Format date
        Time = datetime.datetime.fromtimestamp(ts).strftime('%H:%M:%S')  # Format time
        
        print(f"Images saved for Enrollment: {enrollment}, Name: {name} on {Date} at {Time}")

    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage:
take_img("4", "Dipak Das")

Starting image capture. Press 'q' to quit.
Image 1 saved for Enrollment: 4, Name: Dipak Das
Image 2 saved for Enrollment: 4, Name: Dipak Das
Image 3 saved for Enrollment: 4, Name: Dipak Das
Image 4 saved for Enrollment: 4, Name: Dipak Das
Image 5 saved for Enrollment: 4, Name: Dipak Das
Image 6 saved for Enrollment: 4, Name: Dipak Das
Image 7 saved for Enrollment: 4, Name: Dipak Das
Image 8 saved for Enrollment: 4, Name: Dipak Das
Image 9 saved for Enrollment: 4, Name: Dipak Das
Image 10 saved for Enrollment: 4, Name: Dipak Das
Image 11 saved for Enrollment: 4, Name: Dipak Das
Image 12 saved for Enrollment: 4, Name: Dipak Das
Image 13 saved for Enrollment: 4, Name: Dipak Das
Image 14 saved for Enrollment: 4, Name: Dipak Das
Image 15 saved for Enrollment: 4, Name: Dipak Das
Image 16 saved for Enrollment: 4, Name: Dipak Das
Image 17 saved for Enrollment: 4, Name: Dipak Das
Image 18 saved for Enrollment: 4, Name: Dipak Das
Image 19 saved for Enrollment: 4, Name: Dipak Das
Image 20 saved f

## Testing Model

In [13]:
pip install face_recognition

Defaulting to user installation because normal site-packages is not writeable
  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
  Using cached face_recognition_models-0.3.0-py2.py3-none-any.whl
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import cv2
import datetime
import time
import dlib
import face_recognition
import numpy as np
from IPython.display import display, Image, clear_output

def take_img(enrollment, name):
    """
    Capture images from webcam and save them in a specified directory.
    """
    # Check if the input fields are not empty
    if enrollment == '' or name == '':
        print("Please fill in all fields!")
        return
    
    try:
        # Create a directory for training images if it doesn't exist
        base_dir = 'TrainingImage'
        if not os.path.exists(base_dir):
            os.makedirs(base_dir)

        # Create a subdirectory for the specific enrollment ID
        enrollment_dir = os.path.join(base_dir, enrollment)  # Subfolder named after enrollment ID
        if not os.path.exists(enrollment_dir):
            os.makedirs(enrollment_dir)

        # Open the webcam
        cam = cv2.VideoCapture(0)
        detector = dlib.get_frontal_face_detector()  # Load dlib's face detector
        
        sampleNum = 0  # Initialize sample number
        
        print("Starting image capture. Press 'q' to quit.")
        
        while True:
            ret, img = cam.read()  # Read frame from webcam
            
            if not ret:
                print("Failed to capture image from webcam.")
                break
            
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB
            
            faces = detector(rgb_img)  # Detect faces using dlib
            
            for face in faces:
                x, y, w, h = (face.left(), face.top(), face.width(), face.height())  # Get coordinates of the detected face
                
                # Extract the face region from the image
                face_image = rgb_img[y:y+h, x:x+w]
                
                # Check if the extracted face image is valid (must be RGB)
                if face_image.ndim != 3 or face_image.shape[2] != 3:
                    print("Captured image is not in RGB format. Skipping this capture.")
                    continue
                
                cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)  # Draw rectangle around face
                sampleNum += 1
                
                # Save captured face images in the specific enrollment subfolder
                cv2.imwrite(f"{enrollment_dir}/{name}.{enrollment}.{sampleNum}.jpg", face_image)  # Save only the detected face
                print(f"Image {sampleNum} saved for Enrollment: {enrollment}, Name: {name}")
                
            cv2.imshow('Frame', img)  # Show the frame with detected faces
            
            if cv2.waitKey(1) & 0xFF == ord('q'):  # Break on 'q' key press
                break
            
            if sampleNum >= 70:  # Stop after capturing a certain number of samples
                break

        cam.release()  # Release the webcam
        cv2.destroyAllWindows()  # Close all OpenCV windows

        ts = time.time()  # Get current timestamp
        Date = datetime.datetime.fromtimestamp(ts).strftime('%Y-%m-%d')  # Format date
        Time = datetime.datetime.fromtimestamp(ts).strftime('%H:%M:%S')  # Format time
        
        print(f"Images saved for Enrollment: {enrollment}, Name: {name} on {Date} at {Time}")

    except Exception as e:
        print(f"An error occurred: {e}")

def prepare_training_data(data_folder_path):
    """
    Prepare training data by loading images and extracting face encodings.
    """
    encodings = []
    labels = []
    
    for dir_name in os.listdir(data_folder_path):
        if not dir_name.startswith('.'):  # Skip hidden directories
            label = int(dir_name)  # Convert directory name to label (assuming it's numeric)
            subject_dir_path = os.path.join(data_folder_path, dir_name)
            
            for image_name in os.listdir(subject_dir_path):
                if image_name.endswith(".jpg"):
                    image_path = os.path.join(subject_dir_path, image_name)
                    image = face_recognition.load_image_file(image_path)  # Load image using face_recognition
                    encoding = face_recognition.face_encodings(image)  # Get face encodings
                    
                    if encoding:  # Check if there is at least one encoding
                        encodings.append(encoding[0])  # Add the first encoding to the list
                        labels.append(label)  # Add label to labels list
    
    return encodings, labels

def test_model(encodings, labels):
    """
    Test the model by capturing images from webcam and recognizing faces.
    """
    cam = cv2.VideoCapture(0)  # Open webcam
    
    while True:
        ret, img = cam.read()  # Read frame from webcam
        
        if not ret:
            print("Failed to capture image from webcam.")
            break
        
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB
        
        face_locations = face_recognition.face_locations(rgb_img)  # Find all face locations in the current frame
        face_encodings = face_recognition.face_encodings(rgb_img, face_locations)  # Get encodings for each detected face
        
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            matches = face_recognition.compare_faces(encodings, face_encoding)  # Compare with known encodings
            
            label_text = "Unknown"  # Default label
            
            if True in matches:
                first_match_index = matches.index(True)
                label_text = f"ID: {labels[first_match_index]}"  # Get corresponding label
            
            cv2.rectangle(img, (left, top), (right, bottom), (255, 0, 0), 2)  # Draw rectangle around face
            cv2.putText(img, label_text, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)  # Display label
        
        cv2.imshow('Face Recognition', img)  # Show frame with recognized faces
        
        if cv2.waitKey(1) & 0xFF == ord('q'):  # Break on 'q' key press
            break
    
    cam.release()  # Release webcam
    cv2.destroyAllWindows()  # Close all OpenCV windows

def main():
    action = input("Enter 'capture' to take images or 'recognize' to test recognition: ").strip().lower()
    
    if action == 'capture':
        enrollment_id = input("Enter Enrollment ID: ")
        name = input("Enter Name: ")
        take_img(enrollment_id, name)
    
    elif action == 'recognize':
        data_folder_path = 'TrainingImage'  
        
        print("Preparing training data...")
        encodings, labels = prepare_training_data(data_folder_path) 
        
        print("Testing model...")
        test_model(encodings, labels)

if __name__ == "__main__":
    main()

Enter 'capture' to take images or 'recognize' to test recognition:  recognize


Preparing training data...
Testing model...


In [16]:
pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

  Using cached mysql_connector_python-9.1.0-cp312-cp312-win_amd64.whl.metadata (6.2 kB)
Using cached mysql_connector_python-9.1.0-cp312-cp312-win_amd64.whl (16.1 MB)


## Database Connection

In [ ]:
import os
import cv2
import datetime
import time
import dlib
import face_recognition
import numpy as np
import mysql.connector
from getpass import getpass

def connect_to_database():
    """
    Connect to the MySQL database.
    """
    try:
        connection = mysql.connector.connect(
            host='localhost',  # Change if necessary
            user='root',  # Replace with your MySQL username
            password='ExplorerSoul@2024',  # Replace with your MySQL password
            database='attendance_management_system'  # Replace with your database name
        )
        return connection
    except mysql.connector.Error as err:
        print(f"Error: {err}")
        return None

def login():
    """
    Prompt for username and password to log in.
    """
    username = input("Enter username: ")
    password = getpass("Enter password: ")  # Use getpass for secure input

    connection = connect_to_database()
    if connection:
        cursor = connection.cursor()
        cursor.execute("SELECT * FROM users WHERE username=%s AND password=%s", (username, password))
        result = cursor.fetchone()
        
        cursor.close()
        connection.close()
        
        if result:
            print("Login successful!")
            return True  # Login successful
        else:
            print("Invalid username or password.")
            return False  # Login failed

def take_img(enrollment, name):
    """
    Capture images from webcam and save them in a specified directory.
    """
    if enrollment == '' or name == '':
        print("Please fill in all fields!")
        return
    
    try:
        base_dir = 'TrainingImage'
        if not os.path.exists(base_dir):
            os.makedirs(base_dir)

        enrollment_dir = os.path.join(base_dir, enrollment)
        if not os.path.exists(enrollment_dir):
            os.makedirs(enrollment_dir)

        cam = cv2.VideoCapture(0)
        detector = dlib.get_frontal_face_detector()
        
        sampleNum = 0
        
        print("Starting image capture. Press 'q' to quit.")
        
        while True:
            ret, img = cam.read()
            
            if not ret:
                print("Failed to capture image from webcam.")
                break
            
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            faces = detector(rgb_img)
            
            for face in faces:
                x, y, w, h = (face.left(), face.top(), face.width(), face.height())
                
                face_image = rgb_img[y:y+h, x:x+w]
                
                if face_image.ndim != 3 or face_image.shape[2] != 3:
                    print("Captured image is not in RGB format. Skipping this capture.")
                    continue
                
                cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
                sampleNum += 1
                
                cv2.imwrite(f"{enrollment_dir}/{name}.{enrollment}.{sampleNum}.jpg", face_image)
                print(f"Image {sampleNum} saved for Enrollment: {enrollment}, Name: {name}")
                
            cv2.imshow('Frame', img)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            
            if sampleNum >= 70:
                break

        cam.release()
        cv2.destroyAllWindows()

        ts = time.time()
        Date = datetime.datetime.fromtimestamp(ts).strftime('%Y-%m-%d')
        Time = datetime.datetime.fromtimestamp(ts).strftime('%H:%M:%S')
        
        print(f"Images saved for Enrollment: {enrollment}, Name: {name} on {Date} at {Time}")

    except Exception as e:
        print(f"An error occurred: {e}")

def prepare_training_data(data_folder_path):
    """
    Prepare training data by loading images and extracting face encodings.
    """
    encodings = []
    labels = []
    
    for dir_name in os.listdir(data_folder_path):
        if not dir_name.startswith('.'):
            label = int(dir_name)
            subject_dir_path = os.path.join(data_folder_path, dir_name)
            
            for image_name in os.listdir(subject_dir_path):
                if image_name.endswith(".jpg"):
                    image_path = os.path.join(subject_dir_path, image_name)
                    image = face_recognition.load_image_file(image_path)
                    encoding = face_recognition.face_encodings(image)
                    
                    if encoding:
                        encodings.append(encoding[0])
                        labels.append(label)
    
    return encodings, labels

def test_model(encodings, labels):
    """
    Test the model by capturing images from webcam and recognizing faces.
    """
    cam = cv2.VideoCapture(0)
    
    while True:
        ret, img = cam.read()
        
        if not ret:
            print("Failed to capture image from webcam.")
            break
        
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        face_locations = face_recognition.face_locations(rgb_img)
        face_encodings = face_recognition.face_encodings(rgb_img, face_locations)
        
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            matches = face_recognition.compare_faces(encodings, face_encoding)
            
            label_text = "Unknown"
            
            if True in matches:
                first_match_index = matches.index(True)
                label_text = f"ID: {labels[first_match_index]}"
                
                # Record attendance in the database
                record_attendance(labels[first_match_index], label_text)

            cv2.rectangle(img, (left, top), (right, bottom), (255, 0, 0), 2)
            cv2.putText(img, label_text, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        cv2.imshow('Face Recognition', img)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cam.release()
    cv2.destroyAllWindows()

def record_attendance(enrollment_id, name):
    """
    Record attendance in the MySQL database.
    """
    date_today = datetime.datetime.now().date()  # Get today's date
    time_now = datetime.datetime.now().time()  # Get current time
    
    connection = connect_to_database()
    
    if connection:
        cursor = connection.cursor()
        
        # Insert attendance record into the database
        sql_query = "INSERT INTO attendance (id, name, date, time) VALUES (%s, %s, %s, %s)"
        
        cursor.execute(sql_query, (enrollment_id, name, date_today.strftime('%Y-%m-%d'), time_now.strftime('%H:%M:%S')))
        
        connection.commit()  # Commit changes to the database
        
        cursor.close()
        connection.close()
        
        print(f"Attendance recorded for Student ID: {enrollment_id} ({name}) on {date_today} at {time_now}")

def main():
    action = input("Enter 'login' to access the system or 'exit' to quit: ").strip().lower()
    
    if action == 'login':
        if login():
            action_type = input("Enter 'capture' to take images or 'recognize' to test recognition: ").strip().lower()

            if action_type == 'capture':
                enrollment_id = input("Enter Enrollment ID: ")
                name = input("Enter Name: ")
                take_img(enrollment_id, name)

            elif action_type == 'recognize':
                data_folder_path = 'TrainingImage'
                
                print("Preparing training data...")
                encodings, labels = prepare_training_data(data_folder_path) 
                
                print("Testing model...")
                test_model(encodings, labels)

if __name__ == "__main__":
    main()

In [18]:
import os
import cv2
import datetime
import dlib
import face_recognition
import numpy as np
import mysql.connector
import csv

def connect_to_database():
    """Connect to the MySQL database."""
    try:
        connection = mysql.connector.connect(
            host='localhost',  # Change if necessary
            user='root',  # Replace with your MySQL username
            password='ExplorerSoul@2024',  # Replace with your MySQL password
            database='attendance_management_system'  # Replace with your database name
        )
        return connection
    except mysql.connector.Error as err:
        print(f"Error: {err}")
        return None

def take_img(enrollment, name):
    """Capture images from webcam and save them in a specified directory."""
    if enrollment == '' or name == '':
        print("Please fill in all fields!")
        return
    
    try:
        base_dir = 'TrainingImage'
        if not os.path.exists(base_dir):
            os.makedirs(base_dir)

        enrollment_dir = os.path.join(base_dir, enrollment)
        if not os.path.exists(enrollment_dir):
            os.makedirs(enrollment_dir)

        cam = cv2.VideoCapture(0)
        detector = dlib.get_frontal_face_detector()
        
        sampleNum = 0
        
        print("Starting image capture. Press 'q' to quit.")
        
        while True:
            ret, img = cam.read()
            
            if not ret:
                print("Failed to capture image from webcam.")
                break
            
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            faces = detector(rgb_img)
            
            for face in faces:
                x, y, w, h = (face.left(), face.top(), face.width(), face.height())
                
                face_image = rgb_img[y:y+h, x:x+w]
                
                if face_image.ndim != 3 or face_image.shape[2] != 3:
                    print("Captured image is not in RGB format. Skipping this capture.")
                    continue
                
                cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
                sampleNum += 1
                
                cv2.imwrite(f"{enrollment_dir}/{name}.{enrollment}.{sampleNum}.jpg", face_image)
                print(f"Image {sampleNum} saved for Enrollment: {enrollment}, Name: {name}")
                
            cv2.imshow('Frame', img)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            
            if sampleNum >= 70:
                break

        cam.release()
        cv2.destroyAllWindows()

    except Exception as e:
        print(f"An error occurred: {e}")

def prepare_training_data(data_folder_path):
    """Prepare training data by loading images and extracting face encodings."""
    encodings = []
    labels = []
    
    for dir_name in os.listdir(data_folder_path):
        if not dir_name.startswith('.'):
            label = dir_name  # Assuming dir_name is the enrollment ID
            subject_dir_path = os.path.join(data_folder_path, dir_name)
            
            for image_name in os.listdir(subject_dir_path):
                if image_name.endswith(".jpg"):
                    image_path = os.path.join(subject_dir_path, image_name)
                    image = face_recognition.load_image_file(image_path)
                    encoding = face_recognition.face_encodings(image)
                    
                    if encoding:
                        encodings.append(encoding[0])  # Take the first encoding
                        labels.append(label)  # Use the student ID as the name
    
    return encodings, labels

def test_model(encodings):
    """Test the model by capturing images from webcam and recognizing faces."""
    cam = cv2.VideoCapture(0)
    
    while True:
        ret, img = cam.read()
        
        if not ret:
            print("Failed to capture image from webcam.")
            break
        
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        face_locations = face_recognition.face_locations(rgb_img)
        face_encodings = face_recognition.face_encodings(rgb_img, face_locations)
        
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            matches = face_recognition.compare_faces(encodings, face_encoding)
            
            label_text = "Unknown"
            
            if True in matches:
                first_match_index = matches.index(True)
                label_text = f"ID: {labels[first_match_index]}"
                
                # Record attendance in the database and CSV file
                record_attendance(labels[first_match_index], label_text)

            cv2.rectangle(img, (left, top), (right, bottom), (255, 0, 0), 2)
            cv2.putText(img, label_text, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        cv2.imshow('Face Recognition', img)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cam.release()
    cv2.destroyAllWindows()

def record_attendance(enrollment_id, name):
    """Record attendance in the MySQL database and save it to a CSV file."""
    date_today = datetime.datetime.now().date()  # Get today's date
    time_now = datetime.datetime.now().time()  # Get current time
    
    connection = connect_to_database()
    
    if connection:
        cursor = connection.cursor()
        
        # Insert attendance record into the database
        sql_query = "INSERT INTO attendance (id, name, date, time) VALUES (%s, %s, %s, %s)"
        
        cursor.execute(sql_query, (enrollment_id, name,
                                    date_today.strftime('%Y-%m-%d'),
                                    time_now.strftime('%H:%M:%S')))
        
        connection.commit()  # Commit changes to the database
        
        cursor.close()
        connection.close()
        
        print(f"Attendance recorded for Student ID: {enrollment_id} ({name}) on {date_today} at {time_now}")

    # Save attendance to CSV file
    save_attendance_to_csv(enrollment_id, name)

def save_attendance_to_csv(enrollment_id, name):
    """Save attendance record to a CSV file."""
    attendance_folder = 'attendance'
    
    # Create attendance folder if it doesn't exist
    if not os.path.exists(attendance_folder):
        os.makedirs(attendance_folder)

    csv_file_path = os.path.join(attendance_folder, 'attendance.csv')

    # Check if the CSV file exists to determine whether to write headers or not
    file_exists = os.path.isfile(csv_file_path)

    with open(csv_file_path, mode='a', newline='') as csvfile:
        fieldnames = ['ID', 'Name', 'Date', 'Time']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        # Write header only once
        if not file_exists:
            writer.writeheader()

        writer.writerow({
            'ID': enrollment_id,
            'Name': name,
            'Date': datetime.datetime.now().strftime('%Y-%m-%d'),
            'Time': datetime.datetime.now().strftime('%H:%M:%S')
        })

def main():
    """Main function to run the automatic attendance system."""
    
    subject_name = input("Enter Subject Name for Attendance: ")
    
    # Prepare training data from existing images
    data_folder_path = 'TrainingImage'
    
    print("Preparing training data...")
    encodings, labels = prepare_training_data(data_folder_path) 
    
    print("Testing model...")
    test_model(encodings)  # Call your test_model function here

if __name__ == "__main__":
    main()


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\Python312\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Temp\ipykernel_11972\2618681751.py", line 192, in on_login
    if login(subject_name):
       ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Temp\ipykernel_11972\2618681751.py", line 33, in login
    cursor.execute("SELECT * FROM users WHERE subject=%s AND password=%s", (subject, default_password))
  File "C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\mysql\connector\cursor.py", line 537, in execute
    self._handle_result(self._connection.cmd_query(stmt))
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\mysql\connector\opentelemetry\context_propagation.py", line 97, in wrapper
    return method(cnx, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
import os
import cv2
import datetime
import dlib
import face_recognition
import numpy as np
import mysql.connector

def connect_to_database():
    """Connect to the MySQL database."""
    try:
        connection = mysql.connector.connect(
            host='localhost',  # Change if necessary
            user='root',  # Replace with your MySQL username
            password='ExplorerSoul@2024',  # Replace with your MySQL password
            database='attendance_management_system'  # Replace with your database name
        )
        return connection
    except mysql.connector.Error as err:
        print(f"Error: {err}")
        return None

def take_img(enrollment, name):
    """Capture images from webcam and save them in a specified directory."""
    if enrollment == '' or name == '':
        print("Please fill in all fields!")
        return
    
    try:
        base_dir = 'TrainingImage'
        if not os.path.exists(base_dir):
            os.makedirs(base_dir)

        enrollment_dir = os.path.join(base_dir, enrollment)
        if not os.path.exists(enrollment_dir):
            os.makedirs(enrollment_dir)

        cam = cv2.VideoCapture(0)
        detector = dlib.get_frontal_face_detector()
        
        sampleNum = 0
        
        print("Starting image capture. Press 'q' to quit.")
        
        while True:
            ret, img = cam.read()
            
            if not ret:
                print("Failed to capture image from webcam.")
                break
            
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            faces = detector(rgb_img)
            
            for face in faces:
                x, y, w, h = (face.left(), face.top(), face.width(), face.height())
                
                face_image = rgb_img[y:y+h, x:x+w]
                
                if face_image.ndim != 3 or face_image.shape[2] != 3:
                    print("Captured image is not in RGB format. Skipping this capture.")
                    continue
                
                cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
                sampleNum += 1
                
                cv2.imwrite(f"{enrollment_dir}/{name}.{enrollment}.{sampleNum}.jpg", face_image)
                print(f"Image {sampleNum} saved for Enrollment: {enrollment}, Name: {name}")
                
            cv2.imshow('Frame', img)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            
            if sampleNum >= 70:
                break

        cam.release()
        cv2.destroyAllWindows()

    except Exception as e:
        print(f"An error occurred: {e}")

def prepare_training_data(data_folder_path):
    """Prepare training data by loading images and extracting face encodings."""
    encodings = []
    labels = []
    
    for dir_name in os.listdir(data_folder_path):
        if not dir_name.startswith('.'):
            label = dir_name  # Assuming dir_name is the enrollment ID
            subject_dir_path = os.path.join(data_folder_path, dir_name)
            
            for image_name in os.listdir(subject_dir_path):
                if image_name.endswith(".jpg"):
                    image_path = os.path.join(subject_dir_path, image_name)
                    image = face_recognition.load_image_file(image_path)
                    encoding = face_recognition.face_encodings(image)
                    
                    if encoding:
                        encodings.append(encoding[0])
                        labels.append(label)
    
    return encodings, labels

def test_model(encodings, labels):
    """Test the model by capturing images from webcam and recognizing faces."""
    cam = cv2.VideoCapture(0)
    
    while True:
        ret, img = cam.read()
        
        if not ret:
            print("Failed to capture image from webcam.")
            break
        
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        face_locations = face_recognition.face_locations(rgb_img)
        face_encodings = face_recognition.face_encodings(rgb_img, face_locations)
        
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            matches = face_recognition.compare_faces(encodings, face_encoding)
            
            label_text = "Unknown"
            
            if True in matches:
                first_match_index = matches.index(True)
                label_text = f"ID: {labels[first_match_index]}"
                
                # Record attendance in the database
                record_attendance(labels[first_match_index], label_text)

            cv2.rectangle(img, (left, top), (right, bottom), (255, 0, 0), 2)
            cv2.putText(img, label_text, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        cv2.imshow('Face Recognition', img)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cam.release()
    cv2.destroyAllWindows()

def record_attendance(enrollment_id, name):
    """Record attendance in the MySQL database."""
    date_today = datetime.datetime.now().date()  # Get today's date
    time_now = datetime.datetime.now().time()  # Get current time
    
    connection = connect_to_database()
    
    if connection:
        cursor = connection.cursor()
        
        # Insert attendance record into the database
        sql_query = "INSERT INTO attendance (id, name, date, time) VALUES (%s, %s, %s, %s)"
        
        cursor.execute(sql_query, (enrollment_id, name, date_today.strftime('%Y-%m-%d'), time_now.strftime('%H:%M:%S')))
        
        connection.commit()  # Commit changes to the database
        
        cursor.close()
        connection.close()
        
        print(f"Attendance recorded for Student ID: {enrollment_id} ({name}) on {date_today} at {time_now}")

def main():
    """Main function to run the automatic attendance system."""
    
    subject_name = input("Enter Subject Name for Attendance: ")
    
    # Prepare training data from existing images
    data_folder_path = 'TrainingImage'
    
    print("Preparing training data...")
    encodings, labels = prepare_training_data(data_folder_path) 
    
    print("Testing model...")
    test_model(encodings, labels)  # Pass both encodings and labels

if __name__ == "__main__":
    main()

Enter Subject Name for Attendance:  english


Preparing training data...
